# 01 · Configuração do banco (PostgreSQL + dbt)

Prepara o PostgreSQL no Colab ou localmente. As células da **seção 1** devem ser copiadas no início dos notebooks de ETL, ELT e análises.

## 1. Configuração

### 1.1 Ambiente

In [ ]:
try:
    from google.colab import drive
    NO_COLAB = True
except ImportError:
    NO_COLAB = False

print("Rodando no Colab?", NO_COLAB)

### 1.2 Repositório e dados

No Colab, clonamos o repositório, que já traz os dados em `dados/` (as bases do SINAN como `.csv.zip` oficiais e as fontes externas; ver `dados/README.md`). O Drive é usado só para guardar o backup do banco (seção 3).

In [ ]:
import os

if NO_COLAB:
    BRANCH = 'main'
    PASTA_REPO = '/content/projeto-integracao-eq9'

    if not os.path.exists(PASTA_REPO):
        !git clone -q -b {BRANCH} https://github.com/joaocm1804/projeto-integracao-eq9.git {PASTA_REPO}
    %cd {PASTA_REPO}/notebooks

    drive.mount('/content/drive')

# Os dados vêm do próprio repositório (relativo à pasta notebooks/)
PASTA_DADOS = '../dados/sinan_chikungunya'
PASTA_EXTERNOS = '../dados/externos'

print("Dados em:", os.path.abspath(PASTA_DADOS))

### 1.3 Bibliotecas

In [ ]:
%pip install -q -r ../requirements.txt

### 1.4 Instalando o PostgreSQL (só no Colab)

In [ ]:
if NO_COLAB:
    !apt-get -qq update > /dev/null
    !apt-get -qq install -y postgresql > /dev/null
    !service postgresql start
    !sudo -u postgres psql -c "ALTER USER postgres PASSWORD 'postgres';"
    print("PostgreSQL rodando no Colab.")
else:
    print("Usando o PostgreSQL local.")

### 1.5 Conexão

Os dados ficam em variáveis de ambiente porque o dbt (`profiles.yml`) também lê elas. Localmente, vêm do arquivo `.env`, que não vai para o GitHub.

In [ ]:
from dotenv import load_dotenv

if NO_COLAB:
    os.environ['PG_HOST'] = 'localhost'
    os.environ['PG_PORT'] = '5432'
    os.environ['PG_USER'] = 'postgres'
    os.environ['PG_PASSWORD'] = 'postgres'
    os.environ['PG_DBNAME'] = 'chikungunya_dw'
else:
    load_dotenv('../.env')

usuario_db = os.environ['PG_USER']
senha_db = os.environ['PG_PASSWORD']
host_db = os.environ['PG_HOST']
porta_db = os.environ['PG_PORT']
nome_banco_db = os.environ['PG_DBNAME']

### 1.6 Criando o banco

`CREATE DATABASE` não roda dentro de transação, por isso o `AUTOCOMMIT`.

In [ ]:
from sqlalchemy import create_engine, text

engine_padrao = create_engine(
    f'postgresql+psycopg2://{usuario_db}:{senha_db}@{host_db}:{porta_db}/postgres',
    isolation_level='AUTOCOMMIT'
)

with engine_padrao.connect() as conn:
    banco_existe = conn.execute(
        text("SELECT 1 FROM pg_database WHERE datname = :nome"), {'nome': nome_banco_db}
    ).scalar()

    if not banco_existe:
        conn.execute(text(f"CREATE DATABASE {nome_banco_db}"))
        print(f"Banco {nome_banco_db} criado.")
    else:
        print(f"Banco {nome_banco_db} já existe.")

### 1.7 Conexão e schemas

| Schema | Conteúdo |
|---|---|
| `raw` | Dados brutos (ELT) |
| `etl_dw` | Esquema estrela do ETL |
| `elt_staging`, `elt_dw` | Criados pelo dbt |

In [ ]:
engine = create_engine(f'postgresql+psycopg2://{usuario_db}:{senha_db}@{host_db}:{porta_db}/{nome_banco_db}')

with engine.begin() as conn:
    conn.execute(text("CREATE SCHEMA IF NOT EXISTS raw"))
    conn.execute(text("CREATE SCHEMA IF NOT EXISTS etl_dw"))

print("Conexão criada.")

## 2. Teste

### 2.1 Gravando uma amostra

In [ ]:
import pandas as pd

df_amostra = pd.read_csv(f'{PASTA_DADOS}/CHIKBR23.csv.zip', dtype=str, nrows=1000)  # o pandas lê o zip direto
df_amostra.to_sql('amostra_teste', con=engine, schema='raw', if_exists='replace', index=False)

with engine.connect() as conn:
    total = conn.execute(text("SELECT COUNT(*) FROM raw.amostra_teste")).scalar()

print(f"Linhas gravadas: {total}")

### 2.2 dbt

Deve terminar com **All checks passed!**

In [ ]:
!cd ../dbt_chikungunya && dbt debug --profiles-dir .

## 3. Backup

No Colab o banco é apagado ao fechar a sessão. Salvamos com `pg_dump` e restauramos com `pg_restore`.

In [ ]:
if NO_COLAB:
    ARQUIVO_BACKUP = '/content/drive/MyDrive/projeto_integracao/backup_teste.dump'
else:
    ARQUIVO_BACKUP = '../backup_teste.dump'

!PGPASSWORD={senha_db} pg_dump -h {host_db} -p {porta_db} -U {usuario_db} -Fc -n raw -f {ARQUIVO_BACKUP} {nome_banco_db}

with engine.begin() as conn:
    conn.execute(text("DROP TABLE raw.amostra_teste"))

!PGPASSWORD={senha_db} pg_restore -h {host_db} -p {porta_db} -U {usuario_db} --clean --if-exists --no-owner -d {nome_banco_db} {ARQUIVO_BACKUP}

with engine.connect() as conn:
    total = conn.execute(text("SELECT COUNT(*) FROM raw.amostra_teste")).scalar()

print(f"Linhas após restaurar: {total}")

### 3.1 Limpeza

In [ ]:
with engine.begin() as conn:
    conn.execute(text("DROP TABLE IF EXISTS raw.amostra_teste"))

os.remove(ARQUIVO_BACKUP)
print("Banco pronto.")